Build a Tool-Calling Research Agent with LangChain

This demo will cover building AI Agents with the legacy LangChain AgentExecutor. These are fine for getting started, but for working with more advanced agents and having more finer control, LangChain recommends to use LangGraph, which we cover in other courses.

In [1]:
import os
from dotenv import load_dotenv,find_dotenv
load_dotenv()

True

In [2]:
OPENAI_API_KEY = os.environ["OPENAI_API_KEY"]
TAVILY_API_KEY= os.environ['TAVILY_API_KEY']
WEATHER_API_KEY= os.environ['WEATHER_API_KEY'] 

Create Tools

## Create Tools

![image.png](attachment:b7b1509f-534a-4a18-8aac-f9d1199065a0.png)
- Web Search tool with information extraction

Other toll we will create is : Weather tool
Web Search tool with information extraction
Other toll we will create is : Weather tool

In [3]:
from langchain_core.tools import tool
from markitdown import MarkItDown
from langchain_community.tools.tavily_search import TavilySearchResults
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, TimeoutError
import requests
import json

/var/folders/y4/j6_cks6j2fb8p7zcm0hyft6r0000gn/T/ipykernel_63815/3083136056.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools.tavily_search import TavilySearchResults


In [4]:
tavily_tool = TavilySearchResults(max_results=5,
                                  search_depth='advanced',
                                  include_answer=False,
                                  include_raw_content=True)
# certain websites will not allow you to crawl them unless you specify a user-agent.
# Here you're pretending to be Google Chrome on Windows. This makes your request look like a real browser
session = requests.Session()

/var/folders/y4/j6_cks6j2fb8p7zcm0hyft6r0000gn/T/ipykernel_63815/1365255516.py:1: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the `langchain-tavily package and should be used instead. To use it run `pip install -U `langchain-tavily` and import as `from `langchain_tavily import TavilySearch``.
  tavily_tool = TavilySearchResults(max_results=5,


In [5]:
# certain websites will not allow you to crawl them unless you specify a user-agent.
# Here you're pretending to be Google Chrome on Windows. This makes your request look like a real browser
session = requests.Session()

session.headers.update({
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/142.0.0.0 Safari/537.36",
    "Accept-Language": "en-US,en;q=0.9",
    "Accept-Encoding": "gzip, deflate, br"
})

md = MarkItDown(requests_session=session)

In [6]:
@tool
def search_web_extract_info(query: str) -> list:
    """Search the web for a query and extracts useful information from the search links."""
    print('Calling web search tool')
    results = tavily_tool.invoke(query)
    docs = []

    def extract_content(url):
        """Helper function to extract content from a URL."""
        extracted_info = md.convert(url)
        text_title = extracted_info.title.strip()
        text_content = extracted_info.text_content.strip()
        return text_title + '\n' + text_content
    # parallelize execution of different urls
    with ThreadPoolExecutor() as executor:
        for result in tqdm(results):
            try:
                future = executor.submit(extract_content, result['url'])
                # Wait for up to 15 seconds for the task to complete
                content = future.result(timeout=15)
                docs.append(content)
            except TimeoutError:
                print(f"Extraction timed out for url: {result['url']}")
            except Exception as e:
                print(f"Error extracting from url: {result['url']} - {e}")

    return docs


@tool
def get_weather(query: str) -> list:
    """Search weatherapi to get the current weather of the queried location."""
    print('Calling weather tool')
    base_url = "http://api.weatherapi.com/v1/current.json"
    complete_url = f"{base_url}?key={WEATHER_API_KEY}&q={query}"

    response = requests.get(complete_url)
    data = response.json()
    if data.get("location"):
        return data
    else:
        return "Weather Data Not Found"

Test Tool Calling with LLM

In [7]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o", temperature=0)

tools = [search_web_extract_info, get_weather]

llm_with_tools = llm.bind_tools(tools)

In [8]:
prompt = "Get details of SBI bank earnings call Q4 2025"

response = llm_with_tools.invoke(prompt)

response.tool_calls

[{'name': 'search_web_extract_info',
  'args': {'query': 'SBI bank earnings call Q4 2025 details'},
  'id': 'call_NCs912Iu4PaWqODeAzGeBWMq',
  'type': 'tool_call'}]

In [9]:
prompt = "how is the weather in Bangalore today"
response = llm_with_tools.invoke(prompt)
response.tool_calls

[{'name': 'get_weather',
  'args': {'query': 'Bangalore'},
  'id': 'call_pDaP2plrPEDhmvcrOnetAa4f',
  'type': 'tool_call'}]

Build and Test AI Agent

Build and Test AI Agent

Now that we have defined the tools and the LLM, we can create the agent. We will be using a tool calling agent to bind the tools to the agent with a prompt. We will also add in the capability to store historical conversations as memory

In [10]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

SYS_PROMPT = """Act as a helpful assistant.
                You run in a loop of Thought, Action, PAUSE, Observation.
                At the end of the loop, you output an Answer.
                Use Thought to describe your thoughts about the question you have been asked.
                Use Action to run one of the actions available to you - then return PAUSE.
                Observation will be the result of running those actions.
                Repeat till you get to the answer for the given user query.

                Use the following workflow format:
                  Question: the input task you must solve
                  Thought: you should always think about what to do
                  Action: the action to take which can be any of the following:
                            - break it into smaller steps if needed
                            - see if you can answer the given task with your trained knowledge
                            - call the most relevant tools at your disposal mentioned below in case you need more information
                  Action Input: the input to the action
                  Observation: the result of the action
                  ... (this Thought/Action/Action Input/Observation can repeat N times)
                  Thought: I now know the final answer
                  Final Answer: the final answer to the original input question

                Tools at your disposal to perform tasks as needed:
                  - get_weather: whenever user asks get the weather of a place.
                  - search_web_extract_info: whenever user asks for specific information or if you don't know the answer.
             """
# We have " Tools at your disposal to perform tasks as needed:" this is not need in this case because gpt-4o has native tool calling support.
prompt_template = ChatPromptTemplate.from_messages(
    [
        ("system", SYS_PROMPT),
        ("human", "{query}"),
        MessagesPlaceholder(variable_name="agent_scratchpad"),
    ]
)

prompt_template.messages

[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template="Act as a helpful assistant.\n                You run in a loop of Thought, Action, PAUSE, Observation.\n                At the end of the loop, you output an Answer.\n                Use Thought to describe your thoughts about the question you have been asked.\n                Use Action to run one of the actions available to you - then return PAUSE.\n                Observation will be the result of running those actions.\n                Repeat till you get to the answer for the given user query.\n\n                Use the following workflow format:\n                  Question: the input task you must solve\n                  Thought: you should always think about what to do\n                  Action: the action to take which can be any of the following:\n                            - break it into smaller steps if needed\n                            - see if you can

Now, we can initalize the agent with the LLM, the prompt, and the tools.

The agent is responsible for taking in input and deciding what actions to take.

REMEMBER the Agent does not execute those actions - that is done by the AgentExecutor

Note that we are passing in the model `llm`, not `llm_with_tools`.

That is because `create_tool_calling_agent` will call `.bind_tools` for us under the hood.

This should ideally be used with an LLM which supports tool \ function calling

In [13]:
# create_tool_calling_agent is LangChain built-in function which enable us to create an Agnet that can use tools.

##from langchain.agents import create_tool_calling_agent
from langchain.agents import create_agent

In [14]:
llm = ChatOpenAI(model="gpt-4o", temperature=0)

tools = [search_web_extract_info, get_weather]

In [17]:
agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt=SYS_PROMPT
)

Finally, we combine the agent (the brains) with the tools inside the AgentExecutor (which will repeatedly call the agent and execute tools).

In [ ]:
# from langchain.agents import AgentExecutor

# agent_executor = AgentExecutor(agent=agent,
#                                tools=tools,
#                                early_stopping_method='force',
#                                max_iterations=10)


In [ ]:
query = """Summarize the key points discussed in ICICI's Q1 2026 earnings call"""

response = llm.invoke(query)

print(response.content)
query_sbi = """Summarize the key points discussed in SBI bank Q1 2026 earnings call"""

I'm sorry, but I don't have access to real-time data or specific future events, including ICICI's Q1 2026 earnings call. However, I can help you understand what typically happens during an earnings call and what key points are usually discussed. 

During an earnings call, a company like ICICI would typically cover:

1. **Financial Performance**: Discussion of the quarterly financial results, including revenue, net income, earnings per share, and other key financial metrics.

2. **Business Segments**: Performance of different business segments or divisions, highlighting areas of growth or concern.

3. **Market Conditions**: Commentary on the economic environment and how it is impacting the business.

4. **Strategic Initiatives**: Updates on any strategic initiatives, such as new product launches, partnerships, or market expansions.

5. **Guidance**: Forward-looking statements regarding future performance expectations, including any revisions to previous guidance.

6. **Q&A Session**: An

In [19]:
result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": query
        }
    ]
})

Calling web search tool


 40%|████      | 2/5 [00:01<00:01,  2.09it/s]

Error extracting from url: https://www.investing.com/news/transcripts/earnings-call-transcript-icici-bank-posts-strong-q1-2026-growth-shares-slip-after-hours-93CH-4799621 - 403 Client Error: Forbidden for url: https://www.investing.com/news/transcripts/earnings-call-transcript-icici-bank-posts-strong-q1-2026-growth-shares-slip-after-hours-93CH-4799621


100%|██████████| 5/5 [00:01<00:00,  3.28it/s]

Error extracting from url: https://www.marketscreener.com/news/transcript-icici-bank-limited-q1-2027-earnings-call-jul-18-2026-ce7f50d3db80f123 - 403 Client Error: Forbidden for url: https://www.marketscreener.com/news/transcript-icici-bank-limited-q1-2027-earnings-call-jul-18-2026-ce7f50d3db80f123


In [21]:
result

{'messages': [HumanMessage(content="Summarize the key points discussed in ICICI's Q1 2026 earnings call", additional_kwargs={}, response_metadata={}, id='09d4f639-7cb5-49d2-bb67-6d9dd10ef251'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 25, 'prompt_tokens': 388, 'total_tokens': 413, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_e160a45017', 'id': 'chatcmpl-EDvAdsx9j1Sa38eFWVib9ivG0KzZw', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a010b5-a608-7f13-882e-d8e74155c441-0', tool_calls=[{'name': 'search_web_extract_info', 'args': {'query': 'ICICI Q1 2026 earnings call summary'}, 'id': 'call_0RZjaBT8QGDKjVDVymXDSLll', 'type': 'to

In [22]:
from IPython.display import display, Markdown

In [23]:
##print(result["messages"][-1].content)
display(Markdown(result["messages"][-1].content))

ICICI Bank's Q1 2026 earnings call highlighted several key points:

1. **Financial Performance**: 
   - Profit after tax increased by 15.9% year-over-year to ₹14,805 crore (approximately $1.6 billion).
   - Profit before tax excluding treasury rose by 20.9% year-over-year.
   - Core operating profit grew by 15.6% year-over-year.

2. **Loan and Deposit Growth**:
   - Total loan portfolio grew by 19.6% year-over-year, driven by strong growth in rural, business banking, and corporate lending.
   - Total deposits increased by 14% year-over-year.

3. **Asset Quality**:
   - The net non-performing asset (NPA) ratio was stable at 0.35%.
   - Provisions were lower compared to the previous year.

4. **Income and Expenses**:
   - Fee income surged by 23.5%, contributing to overall non-interest income growth.
   - Operating expenses increased by 10.4% year-over-year.

5. **Capital and Liquidity**:
   - The bank maintained a strong capital position with a common equity tier 1 ratio of 16.19% and a total capital adequacy ratio of 16.84%.

6. **Strategic Focus**:
   - The bank emphasized a 360-degree customer-centric approach and aims to grow profit before tax excluding treasury through strategic initiatives.

Overall, ICICI Bank reported robust financial performance with significant growth in loans and deposits, stable asset quality, and strong capital and liquidity positions.

In [24]:
query_sbi = """Summarize the key points discussed in SBI bank Q1 2026 earnings call"""

In [28]:
result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": query_sbi
        }
    ]
})

Calling web search tool


 40%|████      | 2/5 [00:01<00:01,  2.11it/s]

Error extracting from url: https://www.alphaspread.com/security/nse/sbin/investor-relations/earnings-call/q1-2026 - 'NoneType' object has no attribute 'strip'
Error extracting from url: https://www.investing.com/news/transcripts/earnings-call-transcript-state-bank-of-india-posts-record-q1-2026-profit-93CH-4846577 - 403 Client Error: Forbidden for url: https://www.investing.com/news/transcripts/earnings-call-transcript-state-bank-of-india-posts-record-q1-2026-profit-93CH-4846577


 60%|██████    | 3/5 [00:01<00:00,  2.13it/s]

Error extracting from url: https://sbi.bank.in/documents/17836/53469043/SBI+Press+Release+Q1FY26.pdf/85e94c11-4ba0-58f4-b634-e9e2d7468d7d?t=1754640673989 - 'NoneType' object has no attribute 'strip'


Some characters could not be decoded, and were replaced with REPLACEMENT CHARACTER.
 80%|████████  | 4/5 [00:02<00:00,  1.57it/s]

Error extracting from url: https://www.alphaspread.com/security/nse/sbicard/investor-relations/earnings-call/q1-2026 - 'NoneType' object has no attribute 'strip'


100%|██████████| 5/5 [00:03<00:00,  1.48it/s]

Error extracting from url: https://finance.yahoo.com/quote/SBIN.NS/earnings/SBIN.NS-Q1-2026-earnings_call-341607.html - 404 Client Error: Not Found for url: https://finance.yahoo.com/quote/SBIN.NS/earnings/SBIN.NS-Q1-2026-earnings_call-341607.html


In [29]:
display(Markdown(result["messages"][-1].content))

I couldn't find specific information about the SBI bank Q1 2026 earnings call. You might want to check the official SBI website or financial news outlets for the latest updates on their earnings call. If you have any other questions or need further assistance, feel free to ask!

In [30]:
query_weather = """how is the weather in Bangalore today?
           show detailed statistics
        """

In [31]:
result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": query_weather
        }
    ]
})

Calling weather tool


In [32]:
display(Markdown(result["messages"][-1].content))

The current weather in Bangalore is as follows:

- **Temperature**: 22.4°C (72.4°F)
- **Condition**: Patchy rain nearby
- **Wind**: 11.4 mph (18.4 kph) from the west
- **Humidity**: 75%
- **Cloud Cover**: 100%
- **Pressure**: 1014 mb (29.95 in)
- **Precipitation**: 0.01 mm
- **Visibility**: 10 km (6 miles)
- **UV Index**: 0
- **Feels Like**: 21.6°C (70.9°F)
- **Dew Point**: 17.7°C (63.9°F)
- **Wind Gusts**: Up to 19 mph (30.6 kph)
- **Chance of Rain**: 21%

The weather is currently nighttime in Bangalore.